In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
import numpy as np
from scipy.stats import norm


def getBins(minval, maxval, sigma, alpha, beta, kappa):
    """Remember, bin 0 = below value! last bin mean >= maxval"""
    x = np.linspace(minval, maxval, 255)

    rv = norm(0, sigma)
    pdf = rv.pdf(x)
    pdf /= pdf.max()
    pdf *= alpha
    pdf = pdf.max() * beta - pdf
    cumsum = np.cumsum(pdf)
    cumsum = cumsum / cumsum.max() * kappa
    cumsum -= cumsum[pdf.size // 2]

    return cumsum


def getHorizonLine(vfov, pitch):
    """
    Angles should be in radians.
    """
    ctr = 0.5 - 0.5 * np.tan(pitch) / np.tan(vfov / 2)
    return ctr


In [ ]:
from detectron2.data import DatasetCatalog
from typing import List, Dict
from detectron2.structures import Boxes, Instances
import json
import logging
import random

import numpy as np
import torch
from PIL import Image
from termcolor import colored
import transforms as T

random.seed(140421)
np.random.seed(140421)

# NOTE: perhaps these shouldn't be global variables.
# crops_dataset_cvpr_myDistWider20200403:
pitch_bins = np.linspace(-0.6, 0.6, 255)
pitch_bins_centers = pitch_bins.copy()
pitch_bins_centers[:-1] += np.diff(pitch_bins_centers) / 2
pitch_bins_centers = np.append(pitch_bins_centers, pitch_bins[-1])

horizon_bins = np.linspace(-1.0, 0.95, 255)
horizon_bins_centers = horizon_bins.copy()
horizon_bins_centers[:-1] += np.diff(horizon_bins_centers) / 2
horizon_bins_centers = np.append(horizon_bins_centers, horizon_bins[-1])

roll_bins = getBins(-np.pi / 6, np.pi / 6, 0.5, 0.04, 1.1, np.pi)
roll_bins_centers = roll_bins.copy()
roll_bins_centers[:-1] += np.diff(roll_bins_centers) / 2
roll_bins_centers = np.append(roll_bins_centers, roll_bins[-1])

vfov_bins = np.linspace(0.2389, 1.6, 255)
vfov_bins_centers = vfov_bins.copy()
vfov_bins_centers[:-1] += np.diff(vfov_bins_centers) / 2
vfov_bins_centers = np.append(vfov_bins_centers, vfov_bins[-1])


class CalibDataset:
    def __init__(
        self,
        train: bool = True,
        logger: logging.Logger | None = None,
        json_name: str = "datasets/train_crops_dataset_cvpr_myDistWider.json",
        debug: bool = False,
    ):
        if logger is None:
            self.logger = logging.getLogger("SUN360Horizon")
        else:
            self.logger = logger
        import time

        ts = time.time()
        with open(json_name) as fhdl:
            self.data = json.load(fhdl)

        max_load = -1 if not debug else 100
        self.data = self.data[:max_load]  # Only use 100 examples
        self.logger.info(
            colored(
                "[CalibDataset] Loaded %d images from %s in %.2f seconds."
                % (len(self.data), json_name, time.time() - ts),
                "white",
                "on_blue",
            ),
        )
        random.shuffle(self.data)
        train_load = -2000 if not debug else -50
        if train:
            self.data = self.data[:train_load]
        else:
            self.data = self.data[train_load:]
        self.logger.info(
            "===== %d for the %s set..."
            % (len(self.data), "TRAIN" if train else "VAL"),
        )

    def __getitem__(self, k):
        # ["data/pano360/crops_dataset_cvpr_myDistWider/30723202112.jpg/30723202112.jpg-1.jpg"....
        with open(self.data[k][:-4] + ".json") as fhdl:
            data = json.load(fhdl)
        im_path = self.data[k]
        data = data[0]
        pitch = data["pitch"]  # in radians
        roll = data["roll"]
        vfov = data["vfov"]
        focal_length_35mm_eq = data["focal_length_35mm_eq"]
        horizon = getHorizonLine(vfov, pitch)
        horizon_idx = np.digitize(horizon, horizon_bins)
        pitch_idx = np.digitize(pitch, pitch_bins)
        roll_idx = np.digitize(roll, roll_bins)
        vfov_idx = np.digitize(vfov, vfov_bins)
        horizon_gt = np.zeros((256,), dtype=np.float32)
        pitch_gt = np.zeros((256,), dtype=np.float32)
        roll_gt = np.zeros((256,), dtype=np.float32)
        vfov_gt = np.zeros((256,), dtype=np.float32)
        horizon_gt[horizon_idx] = 1.0
        pitch_gt[pitch_idx] = 1.0
        roll_gt[roll_idx] = 1.0
        vfov_gt[vfov_idx] = 1.0
        horizon_gt, pitch_gt, roll_gt, vfov_gt = map(
            torch.from_numpy, (horizon_gt, pitch_gt, roll_gt, vfov_gt)
        )
        return dict(
            file_name=im_path,
            image_id=im_path,
            pitch=pitch,
            roll=roll,
            horizon=horizon,
            vfov=vfov,
            focal_length_35mm_eq=focal_length_35mm_eq,
            logits=dict(
                gt_horizon=horizon_gt,
                gt_pitch=pitch_gt,
                gt_roll=roll_gt,
                gt_vfov=vfov_gt,
            ),
        )

    def get_all_items(self):
        for i, _ in enumerate(self.data):
            yield self[i]

    def __call__(self):
        return self

    def __len__(self):
        return len(self.data)


debug = False
train_calib = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)
val_calib = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

if debug and len(train_calib) < 500:
    dataset_dicts = list(train_calib.get_all_items())
    for i, d in enumerate(random.sample(dataset_dicts, 3)):
        print(d.keys())
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5)
        out = visualizer.get_output()
        plt.imshow(out.get_image())
        plt.show()

In [ ]:
DatasetCatalog.register("Pano360_train", train_calib)
DatasetCatalog.register("Pano360_val", val_calib)

In [ ]:
import os
from detectron2.engine import CalibTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg
import detectron2.data.transforms as T

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Keypoints/keypoint_rcnn_R_50_FPN_1x.yaml"))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = ("Pano360_train",)
cfg.DATASETS.TEST = ("Pano360_val",)
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Keypoints/keypoint_rcnn_R_50_FPN_1x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 4  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00050  # pick a good LR
cfg.MODEL.META_ARCHITECTURE = "ClassifierRCNN"
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # only has one class (ballon). (see https://detectron2.readthedocs.io/tutorials/datasets.html#update-the-config-for-new-datasets)
cfg.MODEL.ROI_HEADS.NAME = "CombinedClassifierHeads"
cfg.MODEL.PROPOSAL_GENERATOR.NAME = "PrecomputedProposals"  # Disable RPN Network. -- Must add proposals to input data
cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
cfg.MODEL.ROI_BOX_HEAD.NUM_CLASSES_H = 256
cfg.MODEL.ROI_BOX_HEAD.NUM_CONV = 0
cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2
cfg.MODEL.KEYPOINT_ON=False
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
cfg.SOLVER.MAX_ITER=5000
# NOTE: this config means the number of classes, but a few popular unofficial tutorials incorrect uses num_classes+1 here.

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = CalibTrainer(cfg) 
trainer.resume_or_load(resume=True)
trainer.model.backbone.eval()
# trainer.model.proposal_generator.eval()
# trainer.train()

In [ ]:
from detectron2.evaluation import Pano360Evaluator

trainer.model.eval()
trainer.test(cfg, trainer.model, evaluators=Pano360Evaluator())